# Preprocesamiento — Punto 2: Regresión para Consultoría en Desarrollo Global
**Taller 1 · Consultoría Económica con IA** | David Rodríguez · Juan Rueda · 2026

In [906]:
%matplotlib inline
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

NOTEBOOK_DIR = Path().resolve()
DIR_DATOS    = NOTEBOOK_DIR / "Datos"
DIR_VIZS     = NOTEBOOK_DIR / "Visualizaciones"
LOCAL_TRAIN  = DIR_DATOS / "Life Expectancy Data.csv"
URL_TRAIN    = (
    "https://raw.githubusercontent.com/darc-17/Sandbox_HE2_"
    "DavidRodriguez/refs/heads/main/Taller%201/Punto%202.%20"
    "Regresi%C3%B3n%20para%20Consultor%C3%ADa%20en%20Desarrollo%20Global/Life%20Expectancy%20Data.csv"
)

# 1. Carga de Datos y Nombres Variables

In [907]:
if LOCAL_TRAIN.exists():
    df = pd.read_csv(LOCAL_TRAIN)
    print("Fuente: archivo local")
else:
    df = pd.read_csv(URL_TRAIN)
    print("Fuente: URL remota (archivo local no encontrado)")

print(f"Forma inicial: {df.shape[0]:,} obs · {df.shape[1]} variables")
print(f"Tipos de dato:\n{df.dtypes.value_counts().rename('columnas').to_string()}")

Fuente: archivo local
Forma inicial: 2,938 obs · 22 variables
Tipos de dato:
float64    16
int64       4
str         2


In [908]:
# ── Normalizar y renombrar columnas ───────────────────────────────────────────
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

rename_map = {
    'Status':                          'status',
    'Life expectancy':                 'life_expectancy',
    'Adult Mortality':                 'adult_mortality',
    'infant deaths':                   'infant_deaths',
    'Alcohol':                         'alcohol',
    'percentage expenditure':          'health_exp_pct_gdp',
    'Hepatitis B':                     'hepatitis_b',
    'Measles':                         'measles',
    'BMI':                             'bmi',
    'under-five deaths':               'under5_deaths',
    'Polio':                           'polio',
    'Total expenditure':               'gov_health_exp_pct',
    'Diphtheria':                      'diphtheria',
    'HIV/AIDS':                        'hiv_aids',
    'GDP':                             'gdp_per_capita',
    'Population':                      'population',
    'thinness 1-19 years':             'thinness_10_19',
    'thinness 5-9 years':              'thinness_5_9',
    'Income composition of resources': 'hdi_income',
    'Schooling':                       'schooling',
}

df = df.rename(columns=rename_map)
print(f"Columnas renombradas. Total: {df.shape[1]}")
print(list(df.columns))


Columnas renombradas. Total: 22
['Country', 'Year', 'status', 'life_expectancy', 'adult_mortality', 'infant_deaths', 'alcohol', 'health_exp_pct_gdp', 'hepatitis_b', 'measles', 'bmi', 'under5_deaths', 'polio', 'gov_health_exp_pct', 'diphtheria', 'hiv_aids', 'gdp_per_capita', 'population', 'thinness_10_19', 'thinness_5_9', 'hdi_income', 'schooling']


# 2. Revisamos Duplicados y Missings

In [909]:
# ── Duplicados ────────────────────────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f"Filas duplicadas: {n_dup} ({n_dup/len(df)*100:.2f} %)")
if n_dup:
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)))

# ── Missings ──────────────────────────────────────────────────────────────────
miss = (
    pd.DataFrame({
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean() * 100,
    })
    .query("n_missing > 0")
    .sort_values("pct_missing", ascending=False)
)
print(f"\nVariables con valores faltantes: {len(miss)} de {df.shape[1]}")

# ── Columnas con missings ──────────────────────────────────
if not miss.empty:
    print("\nColumnas con missings (n):")
    for col, row in miss.iterrows():
        print(f"  {col:<40} {int(row['n_missing']):>5} ({row['pct_missing']:.2f}%)")


Filas duplicadas: 0 (0.00 %)

Variables con valores faltantes: 14 de 22

Columnas con missings (n):
  population                                 652 (22.19%)
  hepatitis_b                                553 (18.82%)
  gdp_per_capita                             448 (15.25%)
  gov_health_exp_pct                         226 (7.69%)
  alcohol                                    194 (6.60%)
  hdi_income                                 167 (5.68%)
  schooling                                  163 (5.55%)
  thinness_5_9                                34 (1.16%)
  thinness_10_19                              34 (1.16%)
  bmi                                         34 (1.16%)
  polio                                       19 (0.65%)
  diphtheria                                  19 (0.65%)
  life_expectancy                             10 (0.34%)
  adult_mortality                             10 (0.34%)


## 2.1 Países con obs solo en 2013 eliminados

In [910]:
# Explorando con Data Wrangler podemos ver que hay 193 países y los años son
# del 2001 al 2015 (16 años) con 10 más obs en 2013 (países que seguramente solo
# están ese año)

todos_anios = set(df["Year"].unique())
paises_2013 = set(df[df["Year"] == 2013]["Country"].unique())

paises_incompletos = sorted([
    p for p in paises_2013
    if set(df[df["Country"] == p]["Year"].unique()) != todos_anios
])

print(f"Países en 2013 con años faltantes: {len(paises_incompletos)}")
for p in paises_incompletos:
    anios_presentes = sorted(df[df["Country"] == p]["Year"].unique())
    anios_faltantes = sorted(todos_anios - set(anios_presentes))
    print(f"  {p}: faltan {anios_faltantes}")

df = df[~df["Country"].isin(paises_incompletos)].reset_index(drop=True)
print(f"Shape tras eliminar países incompletos: {df.shape}")

Países en 2013 con años faltantes: 10
  Cook Islands: faltan [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015)]
  Dominica: faltan [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015)]
  Marshall Islands: faltan [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015)]
  Monaco: faltan [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64

## 2.2 South Sudan eliminado por muchos missings

In [911]:
df = df[df["Country"] != "South Sudan"].reset_index(drop=True)
print(f"Shape tras eliminar South Sudan: {df.shape}")

Shape tras eliminar South Sudan: (2912, 22)


## 2.3 Dummy develp_status

1 si es desarrollado

In [912]:
df["develop_status"] = (df["status"] == "Developed").astype(int)
df = df.drop(columns="status")
print(df["develop_status"].value_counts())

develop_status
0    2400
1     512
Name: count, dtype: int64


## 2.4 Imputación datos faltantes en GDP per cápita y población 
 

In [913]:
# Antigua y Barbuda

pop_antigua = {
    2000: 75056,  2001: 76213,  2002: 77204,  2003: 78131,
    2004: 79033,  2005: 79941,  2006: 80884,  2007: 81831,
    2008: 82786,  2009: 83761,  2010: 84744,  2011: 85739,
    2012: 86729,  2013: 87706,  2014: 88650,  2015: 89560,
}

mask = df["Country"] == "Antigua and Barbuda"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_antigua)

print(df.loc[mask, ["Country", "Year", "population"]])
df

                Country  Year  population
64  Antigua and Barbuda  2015     89560.0
65  Antigua and Barbuda  2014     88650.0
66  Antigua and Barbuda  2013     87706.0
67  Antigua and Barbuda  2012     86729.0
68  Antigua and Barbuda  2011     85739.0
69  Antigua and Barbuda  2010     84744.0
70  Antigua and Barbuda  2009     83761.0
71  Antigua and Barbuda  2008     82786.0
72  Antigua and Barbuda  2007     81831.0
73  Antigua and Barbuda  2006     80884.0
74  Antigua and Barbuda  2005     79941.0
75  Antigua and Barbuda  2004     79033.0
76  Antigua and Barbuda  2003     78131.0
77  Antigua and Barbuda  2002     77204.0
78  Antigua and Barbuda  2001     76213.0
79  Antigua and Barbuda  2000     75056.0


,Country,Year,life_expectancy,adult_mortality,infant_deaths,alcohol,health_exp_pct_gdp,hepatitis_b,measles,bmi,under5_deaths,polio,gov_health_exp_pct,diphtheria,hiv_aids,gdp_per_capita,population,thinness_10_19,thinness_5_9,hdi_income,schooling,develop_status
0,Afghanistan,2015,65.0,263.0,62,0.01,71.279624,65.0,1154,19.1,83,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1,0
1,Afghanistan,2014,59.9,271.0,64,0.01,73.523582,62.0,492,18.6,86,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0,0
2,Afghanistan,2013,59.9,268.0,66,0.01,73.219243,64.0,430,18.1,89,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9,0
3,Afghanistan,2012,59.5,272.0,69,0.01,78.184215,67.0,2787,17.6,93,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8,0
4,Afghanistan,2011,59.2,275.0,71,0.01,7.097109,68.0,3013,17.2,97,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2907,Zimbabwe,2004,44.3,723.0,27,4.36,0.000000,68.0,31,27.1,42,67.0,7.13,65.0,33.6,454.366654,12777511.0,9.4,9.4,0.407,9.2,0
2908,Zimbabwe,2003,44.5,715.0,26,4.06,0.000000,7.0,998,26.7,41,7.0,6.52,68.0,36.7,453.351155,12633897.0,9.8,9.9,0.418,9.5,0
2909,Zimbabwe,2002,44.8,73.0,25,4.43,0.000000,73.0,304,26.3,40,73.0,6.53,71.0,39.8,57.348340,125525.0,1.2,1.3,0.427,10.0,0
2910,Zimbabwe,2001,45.3,686.0,25,1.72,0.000000,76.0,529,25.9,39,76.0,6.16,75.0,42.1,548.587312,12366165.0,1.6,1.7,0.427,9.8,0


In [914]:
# Bahamas

pop_bahamas = {
    2000: 323835, 2001: 327836, 2002: 331614, 2003: 335618,
    2004: 339782, 2005: 343894, 2006: 348375, 2007: 353334,
    2008: 358120, 2009: 362810, 2010: 367478, 2011: 371729,
    2012: 375469, 2013: 378953, 2014: 382298, 2015: 385346,
}

gdp_bahamas = {
    2000: 27877, 2001: 28144, 2002: 28784, 2003: 28550,
    2004: 28883, 2005: 30422, 2006: 31525, 2007: 32159,
    2008: 32189, 2009: 29910, 2010: 29603, 2011: 29484,
    2012: 30958, 2013: 31104, 2014: 31619, 2015: 32689,
}

mask = df["Country"] == "Bahamas"
df.loc[mask, "population"]   = df.loc[mask, "Year"].map(pop_bahamas)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_bahamas)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


     Country  Year  population  gdp_per_capita
160  Bahamas  2015    385346.0         32689.0
161  Bahamas  2014    382298.0         31619.0
162  Bahamas  2013    378953.0         31104.0
163  Bahamas  2012    375469.0         30958.0
164  Bahamas  2011    371729.0         29484.0
165  Bahamas  2010    367478.0         29603.0
166  Bahamas  2009    362810.0         29910.0
167  Bahamas  2008    358120.0         32189.0
168  Bahamas  2007    353334.0         32159.0
169  Bahamas  2006    348375.0         31525.0
170  Bahamas  2005    343894.0         30422.0
171  Bahamas  2004    339782.0         28883.0
172  Bahamas  2003    335618.0         28550.0
173  Bahamas  2002    331614.0         28784.0
174  Bahamas  2001    327836.0         28144.0
175  Bahamas  2000    323835.0         27877.0


In [915]:
# Bahrain

pop_bahrain = {
    2000: 664614,  2001: 694887,  2002: 730751,  2003: 774152,
    2004: 827080,  2005: 891480,  2006: 967117,  2007: 1051466,
    2008: 1127719, 2009: 1185033, 2010: 1213645, 2011: 1220131,
    2012: 1238103, 2013: 1273506, 2014: 1324530, 2015: 1371851,
}

mask = df["Country"] == "Bahrain"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_bahrain)

print(df.loc[mask, ["Country", "Year", "population"]])


     Country  Year  population
176  Bahrain  2015   1371851.0
177  Bahrain  2014   1324530.0
178  Bahrain  2013   1273506.0
179  Bahrain  2012   1238103.0
180  Bahrain  2011   1220131.0
181  Bahrain  2010   1213645.0
182  Bahrain  2009   1185033.0
183  Bahrain  2008   1127719.0
184  Bahrain  2007   1051466.0
185  Bahrain  2006    967117.0
186  Bahrain  2005    891480.0
187  Bahrain  2004    827080.0
188  Bahrain  2003    774152.0
189  Bahrain  2002    730751.0
190  Bahrain  2001    694887.0
191  Bahrain  2000    664614.0


In [916]:
# Barbados
pop_barbados = {
    2000: 262063, 2001: 263401, 2002: 264748, 2003: 266081,
    2004: 267398, 2005: 268706, 2006: 270016, 2007: 271323,
    2008: 272637, 2009: 273948, 2010: 275243, 2011: 276211,
    2012: 277034, 2013: 277758, 2014: 278414, 2015: 278990,
}

mask = df["Country"] == "Barbados"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_barbados)

print(df.loc[mask, ["Country", "Year", "population"]])


      Country  Year  population
208  Barbados  2015    278990.0
209  Barbados  2014    278414.0
210  Barbados  2013    277758.0
211  Barbados  2012    277034.0
212  Barbados  2011    276211.0
213  Barbados  2010    275243.0
214  Barbados  2009    273948.0
215  Barbados  2008    272637.0
216  Barbados  2007    271323.0
217  Barbados  2006    270016.0
218  Barbados  2005    268706.0
219  Barbados  2004    267398.0
220  Barbados  2003    266081.0
221  Barbados  2002    264748.0
222  Barbados  2001    263401.0
223  Barbados  2000    262063.0


In [917]:
# Bolivia
pop_bolivia = {
    2000: 8592656,  2001: 8751805,  2002: 8911101,  2003: 9071323,
    2004: 9233403,  2005: 9397800,  2006: 9564548,  2007: 9733398,
    2008: 9904057,  2009: 10076343, 2010: 10250111, 2011: 10425321,
    2012: 10601929, 2013: 10779520, 2014: 10958189, 2015: 11138234,
}

gdp_bolivia = {
    2000: 994,  2001: 944,  2002: 894,  2003: 894,
    2004: 945,  2005: 1020, 2006: 1203, 2007: 1353,
    2008: 1688, 2009: 1725, 2010: 1922, 2011: 2305,
    2012: 2561, 2013: 2852, 2014: 3061, 2015: 3019,
}

mask = df["Country"] == "Bolivia (Plurinational State of)"
df.loc[mask, "population"]    = df.loc[mask, "Year"].map(pop_bolivia)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_bolivia)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                              Country  Year  population  gdp_per_capita
304  Bolivia (Plurinational State of)  2015  11138234.0          3019.0
305  Bolivia (Plurinational State of)  2014  10958189.0          3061.0
306  Bolivia (Plurinational State of)  2013  10779520.0          2852.0
307  Bolivia (Plurinational State of)  2012  10601929.0          2561.0
308  Bolivia (Plurinational State of)  2011  10425321.0          2305.0
309  Bolivia (Plurinational State of)  2010  10250111.0          1922.0
310  Bolivia (Plurinational State of)  2009  10076343.0          1725.0
311  Bolivia (Plurinational State of)  2008   9904057.0          1688.0
312  Bolivia (Plurinational State of)  2007   9733398.0          1353.0
313  Bolivia (Plurinational State of)  2006   9564548.0          1203.0
314  Bolivia (Plurinational State of)  2005   9397800.0          1020.0
315  Bolivia (Plurinational State of)  2004   9233403.0           945.0
316  Bolivia (Plurinational State of)  2003   9071323.0         

In [918]:
# Brunei Darussalam
pop_brunei = {
    2000: 333241, 2001: 340117, 2002: 346867, 2003: 353615,
    2004: 360375, 2005: 367159, 2006: 374013, 2007: 380951,
    2008: 388017, 2009: 395150, 2010: 401060, 2011: 406647,
    2012: 411744, 2013: 416656, 2014: 421386, 2015: 425994,
}

mask = df["Country"] == "Brunei Darussalam"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_brunei)

print(df.loc[mask, ["Country", "Year", "population"]])


               Country  Year  population
368  Brunei Darussalam  2015    425994.0
369  Brunei Darussalam  2014    421386.0
370  Brunei Darussalam  2013    416656.0
371  Brunei Darussalam  2012    411744.0
372  Brunei Darussalam  2011    406647.0
373  Brunei Darussalam  2010    401060.0
374  Brunei Darussalam  2009    395150.0
375  Brunei Darussalam  2008    388017.0
376  Brunei Darussalam  2007    380951.0
377  Brunei Darussalam  2006    374013.0
378  Brunei Darussalam  2005    367159.0
379  Brunei Darussalam  2004    360375.0
380  Brunei Darussalam  2003    353615.0
381  Brunei Darussalam  2002    346867.0
382  Brunei Darussalam  2001    340117.0
383  Brunei Darussalam  2000    333241.0


In [919]:
# Côte d'Ivoire
pop_civ = {
    2000: 16454668, 2001: 16913584, 2002: 17370551, 2003: 17838623,
    2004: 18322176, 2005: 18822793, 2006: 19343923, 2007: 19886846,
    2008: 20449606, 2009: 21028818, 2010: 21623383, 2011: 22234453,
    2012: 22863169, 2013: 23513380, 2014: 24188576, 2015: 24888294,
}

gdp_civ = {
    2000: 647,  2001: 638,  2002: 638,  2003: 704,
    2004: 754,  2005: 779,  2006: 804,  2007: 912,
    2008: 1059, 2009: 1057, 2010: 1085, 2011: 1118,
    2012: 1213, 2013: 1409, 2014: 1557, 2015: 1533,
}

mask = df["Country"] == "Côte d'Ivoire"
df.loc[mask, "population"]    = df.loc[mask, "Year"].map(pop_civ)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_civ)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


           Country  Year  population  gdp_per_capita
432  Côte d'Ivoire  2015  24888294.0          1533.0
433  Côte d'Ivoire  2014  24188576.0          1557.0
434  Côte d'Ivoire  2013  23513380.0          1409.0
435  Côte d'Ivoire  2012  22863169.0          1213.0
436  Côte d'Ivoire  2011  22234453.0          1118.0
437  Côte d'Ivoire  2010  21623383.0          1085.0
438  Côte d'Ivoire  2009  21028818.0          1057.0
439  Côte d'Ivoire  2008  20449606.0          1059.0
440  Côte d'Ivoire  2007  19886846.0           912.0
441  Côte d'Ivoire  2006  19343923.0           804.0
442  Côte d'Ivoire  2005  18822793.0           779.0
443  Côte d'Ivoire  2004  18322176.0           754.0
444  Côte d'Ivoire  2003  17838623.0           704.0
445  Côte d'Ivoire  2002  17370551.0           638.0
446  Côte d'Ivoire  2001  16913584.0           638.0
447  Côte d'Ivoire  2000  16454668.0           647.0


In [920]:
# Congo
pop_congo = {
    2000: 3151345, 2001: 3272537, 2002: 3350771, 2003: 3445753,
    2004: 3565554, 2005: 3696393, 2006: 3837409, 2007: 3980430,
    2008: 4113517, 2009: 4281219, 2010: 4462290, 2011: 4609724,
    2012: 4740450, 2013: 4857099, 2014: 4975880, 2015: 5097581,
}

gdp_congo = {
    2000: 1024, 2001: 855,  2002: 906,  2003: 1017,
    2004: 1306, 2005: 1799, 2006: 2104, 2007: 2206,
    2008: 2832, 2009: 2271, 2010: 2947, 2011: 3396,
    2012: 3732, 2013: 3697, 2014: 3601, 2015: 2439,
}

mask = df["Country"] == "Congo"
df.loc[mask, "population"]    = df.loc[mask, "Year"].map(pop_congo)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_congo)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


    Country  Year  population  gdp_per_capita
608   Congo  2015   5097581.0          2439.0
609   Congo  2014   4975880.0          3601.0
610   Congo  2013   4857099.0          3697.0
611   Congo  2012   4740450.0          3732.0
612   Congo  2011   4609724.0          3396.0
613   Congo  2010   4462290.0          2947.0
614   Congo  2009   4281219.0          2271.0
615   Congo  2008   4113517.0          2832.0
616   Congo  2007   3980430.0          2206.0
617   Congo  2006   3837409.0          2104.0
618   Congo  2005   3696393.0          1799.0
619   Congo  2004   3565554.0          1306.0
620   Congo  2003   3445753.0          1017.0
621   Congo  2002   3350771.0           906.0
622   Congo  2001   3272537.0           855.0
623   Congo  2000   3151345.0          1024.0


In [921]:
# Democratic Republic of the Congo
pop_drc = {
    2000: 50507442, 2001: 52132646, 2002: 53750524, 2003: 55343867,
    2004: 56997741, 2005: 58775724, 2006: 60615908, 2007: 62477752,
    2008: 64390664, 2009: 66412044, 2010: 68563038, 2011: 70849311,
    2012: 73254618, 2013: 75789395, 2014: 78403242, 2015: 81035531,
}

gdp_drc = {
    2000: 378, 2001: 143, 2002: 162, 2003: 161,
    2004: 181, 2005: 204, 2006: 238, 2007: 268,
    2008: 307, 2009: 281, 2010: 315, 2011: 365,
    2012: 400, 2013: 431, 2014: 458, 2015: 468,
}

mask = df["Country"] == "Democratic Republic of the Congo"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_drc)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_drc)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                              Country  Year  population  gdp_per_capita
720  Democratic Republic of the Congo  2015  81035531.0           468.0
721  Democratic Republic of the Congo  2014  78403242.0           458.0
722  Democratic Republic of the Congo  2013  75789395.0           431.0
723  Democratic Republic of the Congo  2012  73254618.0           400.0
724  Democratic Republic of the Congo  2011  70849311.0           365.0
725  Democratic Republic of the Congo  2010  68563038.0           315.0
726  Democratic Republic of the Congo  2009  66412044.0           281.0
727  Democratic Republic of the Congo  2008  64390664.0           307.0
728  Democratic Republic of the Congo  2007  62477752.0           268.0
729  Democratic Republic of the Congo  2006  60615908.0           238.0
730  Democratic Republic of the Congo  2005  58775724.0           204.0
731  Democratic Republic of the Congo  2004  56997741.0           181.0
732  Democratic Republic of the Congo  2003  55343867.0         

In [922]:
# Cuba
pop_cuba = {
    2000: 11126420, 2001: 11164680, 2002: 11199440, 2003: 11231160,
    2004: 11260110, 2005: 11285170, 2006: 11304360, 2007: 11316980,
    2008: 11324200, 2009: 11326900, 2010: 11325100, 2011: 11321500,
    2012: 11320600, 2013: 11322400, 2014: 11324800, 2015: 11325060,
}

mask = df["Country"] == "Cuba"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_cuba)

print(df.loc[mask, ["Country", "Year", "population"]])


    Country  Year  population
656    Cuba  2015  11325060.0
657    Cuba  2014  11324800.0
658    Cuba  2013  11322400.0
659    Cuba  2012  11320600.0
660    Cuba  2011  11321500.0
661    Cuba  2010  11325100.0
662    Cuba  2009  11326900.0
663    Cuba  2008  11324200.0
664    Cuba  2007  11316980.0
665    Cuba  2006  11304360.0
666    Cuba  2005  11285170.0
667    Cuba  2004  11260110.0
668    Cuba  2003  11231160.0
669    Cuba  2002  11199440.0
670    Cuba  2001  11164680.0
671    Cuba  2000  11126420.0


In [923]:
# Czechia
pop_czechia = {
    2000: 10266542, 2001: 10246178, 2002: 10201311, 2003: 10200741,
    2004: 10206823, 2005: 10220577, 2006: 10251079, 2007: 10298828,
    2008: 10384603, 2009: 10443975, 2010: 10474408, 2011: 10496084,
    2012: 10510785, 2013: 10514051, 2014: 10525347, 2015: 10546059,
}

gdp_czechia = {
    2000: 6013,  2001: 6375,  2002: 8004,  2003: 9736,
    2004: 11598, 2005: 13285, 2006: 15315, 2007: 18485,
    2008: 22701, 2009: 19735, 2010: 19925, 2011: 21716,
    2012: 19729, 2013: 19915, 2014: 20807, 2015: 17557,
}

mask = df["Country"] == "Czechia"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_czechia)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_czechia)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


     Country  Year  population  gdp_per_capita
688  Czechia  2015  10546059.0         17557.0
689  Czechia  2014  10525347.0         20807.0
690  Czechia  2013  10514051.0         19915.0
691  Czechia  2012  10510785.0         19729.0
692  Czechia  2011  10496084.0         21716.0
693  Czechia  2010  10474408.0         19925.0
694  Czechia  2009  10443975.0         19735.0
695  Czechia  2008  10384603.0         22701.0
696  Czechia  2007  10298828.0         18485.0
697  Czechia  2006  10251079.0         15315.0
698  Czechia  2005  10220577.0         13285.0
699  Czechia  2004  10206823.0         11598.0
700  Czechia  2003  10200741.0          9736.0
701  Czechia  2002  10201311.0          8004.0
702  Czechia  2001  10246178.0          6375.0
703  Czechia  2000  10266542.0          6013.0


In [924]:
# Democratic People's Republic of Korea
pop_dprk = {
    2000: 23665910, 2001: 23815358, 2002: 23943973, 2003: 24090380,
    2004: 24254943, 2005: 24396433, 2006: 24524507, 2007: 24646383,
    2008: 24764618, 2009: 24878332, 2010: 24987258, 2011: 25097874,
    2012: 25211728, 2013: 25329451, 2014: 25451262, 2015: 25575350,
}

gdp_dprk = {
    2000: 462, 2001: 440, 2002: 434, 2003: 461,
    2004: 495, 2005: 519, 2006: 538, 2007: 563,
    2008: 576, 2009: 552, 2010: 573, 2011: 593,
    2012: 612, 2013: 631, 2014: 643, 2015: 654,
}

mask = df["Country"] == "Democratic People's Republic of Korea"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_dprk)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_dprk)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                                   Country  Year  population  gdp_per_capita
704  Democratic People's Republic of Korea  2015  25575350.0           654.0
705  Democratic People's Republic of Korea  2014  25451262.0           643.0
706  Democratic People's Republic of Korea  2013  25329451.0           631.0
707  Democratic People's Republic of Korea  2012  25211728.0           612.0
708  Democratic People's Republic of Korea  2011  25097874.0           593.0
709  Democratic People's Republic of Korea  2010  24987258.0           573.0
710  Democratic People's Republic of Korea  2009  24878332.0           552.0
711  Democratic People's Republic of Korea  2008  24764618.0           576.0
712  Democratic People's Republic of Korea  2007  24646383.0           563.0
713  Democratic People's Republic of Korea  2006  24524507.0           538.0
714  Democratic People's Republic of Korea  2005  24396433.0           519.0
715  Democratic People's Republic of Korea  2004  24254943.0           495.0

In [925]:
# Egypt
pop_egypt = {
    2000: 71371000, 2001: 72948100, 2002: 74533700, 2003: 76145000,
    2004: 77794700, 2005: 79491900, 2006: 81241700, 2007: 83044600,
    2008: 84904500, 2009: 86819300, 2010: 88782500, 2011: 91093000,
    2012: 93161000, 2013: 95333500, 2014: 97528600, 2015: 99597300,
}

gdp_egypt = {
    2000: 1450, 2001: 1324, 2002: 1154, 2003: 1053,
    2004: 1010, 2005: 1114, 2006: 1313, 2007: 1559,
    2008: 1901, 2009: 2185, 2010: 2427, 2011: 2547,
    2012: 2987, 2013: 3048, 2014: 3141, 2015: 3365,
}

mask = df["Country"] == "Egypt"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_egypt)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_egypt)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


    Country  Year  population  gdp_per_capita
800   Egypt  2015  99597300.0          3365.0
801   Egypt  2014  97528600.0          3141.0
802   Egypt  2013  95333500.0          3048.0
803   Egypt  2012  93161000.0          2987.0
804   Egypt  2011  91093000.0          2547.0
805   Egypt  2010  88782500.0          2427.0
806   Egypt  2009  86819300.0          2185.0
807   Egypt  2008  84904500.0          1901.0
808   Egypt  2007  83044600.0          1559.0
809   Egypt  2006  81241700.0          1313.0
810   Egypt  2005  79491900.0          1114.0
811   Egypt  2004  77794700.0          1010.0
812   Egypt  2003  76145000.0          1053.0
813   Egypt  2002  74533700.0          1154.0
814   Egypt  2001  72948100.0          1324.0
815   Egypt  2000  71371000.0          1450.0


In [926]:
# Gambia
pop_gambia = {
    2000: 1317703, 2001: 1364076, 2002: 1411402, 2003: 1459852,
    2004: 1509493, 2005: 1560457, 2006: 1612795, 2007: 1666458,
    2008: 1721570, 2009: 1778131, 2010: 1836163, 2011: 1895689,
    2012: 1956711, 2013: 2019251, 2014: 2083303, 2015: 2148877,
}

gdp_gambia = {
    2000: 601, 2001: 588, 2002: 514, 2003: 481,
    2004: 548, 2005: 572, 2006: 563, 2007: 643,
    2008: 760, 2009: 763, 2010: 787, 2011: 716,
    2012: 727, 2013: 714, 2014: 623, 2015: 661,
}

mask = df["Country"] == "Gambia"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_gambia)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_gambia)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


    Country  Year  population  gdp_per_capita
960  Gambia  2015   2148877.0           661.0
961  Gambia  2014   2083303.0           623.0
962  Gambia  2013   2019251.0           714.0
963  Gambia  2012   1956711.0           727.0
964  Gambia  2011   1895689.0           716.0
965  Gambia  2010   1836163.0           787.0
966  Gambia  2009   1778131.0           763.0
967  Gambia  2008   1721570.0           760.0
968  Gambia  2007   1666458.0           643.0
969  Gambia  2006   1612795.0           563.0
970  Gambia  2005   1560457.0           572.0
971  Gambia  2004   1509493.0           548.0
972  Gambia  2003   1459852.0           481.0
973  Gambia  2002   1411402.0           514.0
974  Gambia  2001   1364076.0           588.0
975  Gambia  2000   1317703.0           601.0


In [927]:
# New Zealand
pop_nz = {
    2000: 3857700, 2001: 3880500, 2002: 3948500, 2003: 4027200,
    2004: 4087500, 2005: 4133900, 2006: 4184600, 2007: 4223800,
    2008: 4259800, 2009: 4302600, 2010: 4350700, 2011: 4384000,
    2012: 4408100, 2013: 4442100, 2014: 4516500, 2015: 4609400,
}

mask = df["Country"] == "New Zealand"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_nz)

print(df.loc[mask, ["Country", "Year", "population"]])


          Country  Year  population
1840  New Zealand  2015   4609400.0
1841  New Zealand  2014   4516500.0
1842  New Zealand  2013   4442100.0
1843  New Zealand  2012   4408100.0
1844  New Zealand  2011   4384000.0
1845  New Zealand  2010   4350700.0
1846  New Zealand  2009   4302600.0
1847  New Zealand  2008   4259800.0
1848  New Zealand  2007   4223800.0
1849  New Zealand  2006   4184600.0
1850  New Zealand  2005   4133900.0
1851  New Zealand  2004   4087500.0
1852  New Zealand  2003   4027200.0
1853  New Zealand  2002   3948500.0
1854  New Zealand  2001   3880500.0
1855  New Zealand  2000   3857700.0


In [928]:
# Grenada
pop_grenada = {
    2000: 102951, 2001: 103595, 2002: 104271, 2003: 104964,
    2004: 105655, 2005: 106346, 2006: 107044, 2007: 107746,
    2008: 108462, 2009: 109194, 2010: 109936, 2011: 110671,
    2012: 111385, 2013: 112059, 2014: 112683, 2015: 113260,
}

mask = df["Country"] == "Grenada"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_grenada)

print(df.loc[mask, ["Country", "Year", "population"]])


      Country  Year  population
1040  Grenada  2015    113260.0
1041  Grenada  2014    112683.0
1042  Grenada  2013    112059.0
1043  Grenada  2012    111385.0
1044  Grenada  2011    110671.0
1045  Grenada  2010    109936.0
1046  Grenada  2009    109194.0
1047  Grenada  2008    108462.0
1048  Grenada  2007    107746.0
1049  Grenada  2006    107044.0
1050  Grenada  2005    106346.0
1051  Grenada  2004    105655.0
1052  Grenada  2003    104964.0
1053  Grenada  2002    104271.0
1054  Grenada  2001    103595.0
1055  Grenada  2000    102951.0


In [929]:
# Iran (Islamic Republic of)
pop_iran = {
    2000: 65544383, 2001: 66350540, 2002: 67235147, 2003: 68168740,
    2004: 69136193, 2005: 70132608, 2006: 71154611, 2007: 72197370,
    2008: 73256191, 2009: 74316748, 2010: 75375696, 2011: 76437422,
    2012: 77495302, 2013: 78515607, 2014: 79476308, 2015: 80416554,
}

gdp_iran = {
    2000: 1612, 2001: 1889, 2002: 1873, 2003: 2260,
    2004: 2726, 2005: 3212, 2006: 3745, 2007: 4756,
    2008: 5625, 2009: 5546, 2010: 6603, 2011: 7951,
    2012: 7933, 2013: 6145, 2014: 5659, 2015: 4917,
}

mask = df["Country"] == "Iran (Islamic Republic of)"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_iran)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_iran)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                         Country  Year  population  gdp_per_capita
1216  Iran (Islamic Republic of)  2015  80416554.0          4917.0
1217  Iran (Islamic Republic of)  2014  79476308.0          5659.0
1218  Iran (Islamic Republic of)  2013  78515607.0          6145.0
1219  Iran (Islamic Republic of)  2012  77495302.0          7933.0
1220  Iran (Islamic Republic of)  2011  76437422.0          7951.0
1221  Iran (Islamic Republic of)  2010  75375696.0          6603.0
1222  Iran (Islamic Republic of)  2009  74316748.0          5546.0
1223  Iran (Islamic Republic of)  2008  73256191.0          5625.0
1224  Iran (Islamic Republic of)  2007  72197370.0          4756.0
1225  Iran (Islamic Republic of)  2006  71154611.0          3745.0
1226  Iran (Islamic Republic of)  2005  70132608.0          3212.0
1227  Iran (Islamic Republic of)  2004  69136193.0          2726.0
1228  Iran (Islamic Republic of)  2003  68168740.0          2260.0
1229  Iran (Islamic Republic of)  2002  67235147.0          18

In [930]:
# Kuwait
pop_kuwait = {
    2000: 2050741, 2001: 2069451, 2002: 2109726, 2003: 2185992,
    2004: 2298401, 2005: 2431234, 2006: 2569679, 2007: 2704909,
    2008: 2836442, 2009: 2975419, 2010: 3126303, 2011: 3303091,
    2012: 3481257, 2013: 3648160, 2014: 3803500, 2015: 3935794,
}

mask = df["Country"] == "Kuwait"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_kuwait)

print(df.loc[mask, ["Country", "Year", "population"]])


     Country  Year  population
1392  Kuwait  2015   3935794.0
1393  Kuwait  2014   3803500.0
1394  Kuwait  2013   3648160.0
1395  Kuwait  2012   3481257.0
1396  Kuwait  2011   3303091.0
1397  Kuwait  2010   3126303.0
1398  Kuwait  2009   2975419.0
1399  Kuwait  2008   2836442.0
1400  Kuwait  2007   2704909.0
1401  Kuwait  2006   2569679.0
1402  Kuwait  2005   2431234.0
1403  Kuwait  2004   2298401.0
1404  Kuwait  2003   2185992.0
1405  Kuwait  2002   2109726.0
1406  Kuwait  2001   2069451.0
1407  Kuwait  2000   2050741.0


In [931]:
# Kyrgyzstan
pop_kyrgyzstan = {
    2000: 4898400, 2001: 4946500, 2002: 4999900, 2003: 5056800,
    2004: 5114100, 2005: 5163300, 2006: 5209500, 2007: 5255100,
    2008: 5308100, 2009: 5379400, 2010: 5447900, 2011: 5514600,
    2012: 5607200, 2013: 5719500, 2014: 5835561, 2015: 5956900,
}

gdp_kyrgyzstan = {
    2000: 280,  2001: 308,  2002: 322,  2003: 380,
    2004: 434,  2005: 476,  2006: 543,  2007: 721,
    2008: 961,  2009: 873,  2010: 880,  2011: 1121,
    2012: 1209, 2013: 1303, 2014: 1280, 2015: 1055,
}

mask = df["Country"] == "Kyrgyzstan"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_kyrgyzstan)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_kyrgyzstan)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


         Country  Year  population  gdp_per_capita
1408  Kyrgyzstan  2015   5956900.0          1055.0
1409  Kyrgyzstan  2014   5835561.0          1280.0
1410  Kyrgyzstan  2013   5719500.0          1303.0
1411  Kyrgyzstan  2012   5607200.0          1209.0
1412  Kyrgyzstan  2011   5514600.0          1121.0
1413  Kyrgyzstan  2010   5447900.0           880.0
1414  Kyrgyzstan  2009   5379400.0           873.0
1415  Kyrgyzstan  2008   5308100.0           961.0
1416  Kyrgyzstan  2007   5255100.0           721.0
1417  Kyrgyzstan  2006   5209500.0           543.0
1418  Kyrgyzstan  2005   5163300.0           476.0
1419  Kyrgyzstan  2004   5114100.0           434.0
1420  Kyrgyzstan  2003   5056800.0           380.0
1421  Kyrgyzstan  2002   4999900.0           322.0
1422  Kyrgyzstan  2001   4946500.0           308.0
1423  Kyrgyzstan  2000   4898400.0           280.0


In [932]:
# Lao People's Democratic Republic
pop_lao = {
    2000: 5323700, 2001: 5417495, 2002: 5511856, 2003: 5604809,
    2004: 5696523, 2005: 5787208, 2006: 5878024, 2007: 5969726,
    2008: 6063205, 2009: 6159383, 2010: 6259001, 2011: 6363227,
    2012: 6473054, 2013: 6587931, 2014: 6707314, 2015: 6830307,
}

gdp_lao = {
    2000: 325,  2001: 324,  2002: 333,  2003: 385,
    2004: 452,  2005: 491,  2006: 614,  2007: 721,
    2008: 905,  2009: 969,  2010: 1141, 2011: 1348,
    2012: 1529, 2013: 1745, 2014: 2021, 2015: 2159,
}

mask = df["Country"] == "Lao People's Democratic Republic"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_lao)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_lao)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                               Country  Year  population  gdp_per_capita
1424  Lao People's Democratic Republic  2015   6830307.0          2159.0
1425  Lao People's Democratic Republic  2014   6707314.0          2021.0
1426  Lao People's Democratic Republic  2013   6587931.0          1745.0
1427  Lao People's Democratic Republic  2012   6473054.0          1529.0
1428  Lao People's Democratic Republic  2011   6363227.0          1348.0
1429  Lao People's Democratic Republic  2010   6259001.0          1141.0
1430  Lao People's Democratic Republic  2009   6159383.0           969.0
1431  Lao People's Democratic Republic  2008   6063205.0           905.0
1432  Lao People's Democratic Republic  2007   5969726.0           721.0
1433  Lao People's Democratic Republic  2006   5878024.0           614.0
1434  Lao People's Democratic Republic  2005   5787208.0           491.0
1435  Lao People's Democratic Republic  2004   5696523.0           452.0
1436  Lao People's Democratic Republic  2003   5604

In [933]:
# Micronesia (Federated States of)
pop_micronesia = {
    2000: 107103, 2001: 107243, 2002: 107173, 2003: 106944,
    2004: 106589, 2005: 106143, 2006: 105683, 2007: 105230,
    2008: 104800, 2009: 104443, 2010: 104185, 2011: 104142,
    2012: 104409, 2013: 105038, 2014: 106031, 2015: 107421,
}

gdp_micronesia = {
    2000: 2175, 2001: 2185, 2002: 2231, 2003: 2253,
    2004: 2246, 2005: 2341, 2006: 2374, 2007: 2427,
    2008: 2476, 2009: 2641, 2010: 2804, 2011: 2949,
    2012: 3101, 2013: 3161, 2014: 3148, 2015: 3211,
}

mask = df["Country"] == "Micronesia (Federated States of)"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_micronesia)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_micronesia)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                               Country  Year  population  gdp_per_capita
1696  Micronesia (Federated States of)  2015    107421.0          3211.0
1697  Micronesia (Federated States of)  2014    106031.0          3148.0
1698  Micronesia (Federated States of)  2013    105038.0          3161.0
1699  Micronesia (Federated States of)  2012    104409.0          3101.0
1700  Micronesia (Federated States of)  2011    104142.0          2949.0
1701  Micronesia (Federated States of)  2010    104185.0          2804.0
1702  Micronesia (Federated States of)  2009    104443.0          2641.0
1703  Micronesia (Federated States of)  2008    104800.0          2476.0
1704  Micronesia (Federated States of)  2007    105230.0          2427.0
1705  Micronesia (Federated States of)  2006    105683.0          2374.0
1706  Micronesia (Federated States of)  2005    106143.0          2341.0
1707  Micronesia (Federated States of)  2004    106589.0          2246.0
1708  Micronesia (Federated States of)  2003    106

In [934]:
# Oman
pop_oman = {
    2000: 2267974, 2001: 2283634, 2002: 2308016, 2003: 2344251,
    2004: 2398552, 2005: 2476715, 2006: 2584286, 2007: 2714129,
    2008: 2835357, 2009: 2912253, 2010: 3082372, 2011: 3365302,
    2012: 3695502, 2013: 4008574, 2014: 4260341, 2015: 4459614,
}

mask = df["Country"] == "Oman"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_oman)

print(df.loc[mask, ["Country", "Year", "population"]])


     Country  Year  population
1920    Oman  2015   4459614.0
1921    Oman  2014   4260341.0
1922    Oman  2013   4008574.0
1923    Oman  2012   3695502.0
1924    Oman  2011   3365302.0
1925    Oman  2010   3082372.0
1926    Oman  2009   2912253.0
1927    Oman  2008   2835357.0
1928    Oman  2007   2714129.0
1929    Oman  2006   2584286.0
1930    Oman  2005   2476715.0
1931    Oman  2004   2398552.0
1932    Oman  2003   2344251.0
1933    Oman  2002   2308016.0
1934    Oman  2001   2283634.0
1935    Oman  2000   2267974.0


In [935]:
# Qatar
pop_qatar = {
    2000: 644989,  2001: 677925,  2002: 711205,  2003: 744682,
    2004: 773332,  2005: 825408,  2006: 972831,  2007: 1208595,
    2008: 1426938, 2009: 1586050, 2010: 1709229, 2011: 1810950,
    2012: 1907117, 2013: 2032641, 2014: 2218372, 2015: 2427331,
}

mask = df["Country"] == "Qatar"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_qatar)

print(df.loc[mask, ["Country", "Year", "population"]])


     Country  Year  population
2064   Qatar  2015   2427331.0
2065   Qatar  2014   2218372.0
2066   Qatar  2013   2032641.0
2067   Qatar  2012   1907117.0
2068   Qatar  2011   1810950.0
2069   Qatar  2010   1709229.0
2070   Qatar  2009   1586050.0
2071   Qatar  2008   1426938.0
2072   Qatar  2007   1208595.0
2073   Qatar  2006    972831.0
2074   Qatar  2005    825408.0
2075   Qatar  2004    773332.0
2076   Qatar  2003    744682.0
2077   Qatar  2002    711205.0
2078   Qatar  2001    677925.0
2079   Qatar  2000    644989.0


In [936]:
# Republic of Korea
pop_korea = {
    2000: 47008111, 2001: 47370164, 2002: 47644736, 2003: 47892330,
    2004: 48082519, 2005: 48184561, 2006: 48438292, 2007: 48683638,
    2008: 49054708, 2009: 49307835, 2010: 49554112, 2011: 49936638,
    2012: 50199853, 2013: 50428893, 2014: 50746659, 2015: 51014947,
}

gdp_korea = {
    2000: 12257, 2001: 11561, 2002: 13165, 2003: 14673,
    2004: 16496, 2005: 19403, 2006: 21743, 2007: 24086,
    2008: 21350, 2009: 19144, 2010: 23079, 2011: 25098,
    2012: 25459, 2013: 27180, 2014: 29253, 2015: 28737,
}

mask = df["Country"] == "Republic of Korea"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_korea)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_korea)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                Country  Year  population  gdp_per_capita
2080  Republic of Korea  2015  51014947.0         28737.0
2081  Republic of Korea  2014  50746659.0         29253.0
2082  Republic of Korea  2013  50428893.0         27180.0
2083  Republic of Korea  2012  50199853.0         25459.0
2084  Republic of Korea  2011  49936638.0         25098.0
2085  Republic of Korea  2010  49554112.0         23079.0
2086  Republic of Korea  2009  49307835.0         19144.0
2087  Republic of Korea  2008  49054708.0         21350.0
2088  Republic of Korea  2007  48683638.0         24086.0
2089  Republic of Korea  2006  48438292.0         21743.0
2090  Republic of Korea  2005  48184561.0         19403.0
2091  Republic of Korea  2004  48082519.0         16496.0
2092  Republic of Korea  2003  47892330.0         14673.0
2093  Republic of Korea  2002  47644736.0         13165.0
2094  Republic of Korea  2001  47370164.0         11561.0
2095  Republic of Korea  2000  47008111.0         12257.0


In [937]:
# Republic of Moldova
pop_moldova = {
    2000: 3639000, 2001: 3626000, 2002: 3613000, 2003: 3602000,
    2004: 3595000, 2005: 3589000, 2006: 3581000, 2007: 3573000,
    2008: 3568000, 2009: 3564000, 2010: 3562000, 2011: 3560000,
    2012: 3560000, 2013: 3559000, 2014: 3557000, 2015: 3555000,
}

gdp_moldova = {
    2000: 354,  2001: 408,  2002: 460,  2003: 548,
    2004: 721,  2005: 831,  2006: 951,  2007: 1232,
    2008: 1695, 2009: 1525, 2010: 1632, 2011: 1971,
    2012: 2047, 2013: 2360, 2014: 2385, 2015: 1820,
}

mask = df["Country"] == "Republic of Moldova"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_moldova)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_moldova)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                  Country  Year  population  gdp_per_capita
2096  Republic of Moldova  2015   3555000.0          1820.0
2097  Republic of Moldova  2014   3557000.0          2385.0
2098  Republic of Moldova  2013   3559000.0          2360.0
2099  Republic of Moldova  2012   3560000.0          2047.0
2100  Republic of Moldova  2011   3560000.0          1971.0
2101  Republic of Moldova  2010   3562000.0          1632.0
2102  Republic of Moldova  2009   3564000.0          1525.0
2103  Republic of Moldova  2008   3568000.0          1695.0
2104  Republic of Moldova  2007   3573000.0          1232.0
2105  Republic of Moldova  2006   3581000.0           951.0
2106  Republic of Moldova  2005   3589000.0           831.0
2107  Republic of Moldova  2004   3595000.0           721.0
2108  Republic of Moldova  2003   3602000.0           548.0
2109  Republic of Moldova  2002   3613000.0           460.0
2110  Republic of Moldova  2001   3626000.0           408.0
2111  Republic of Moldova  2000   363900

In [938]:
# Saint Lucia
pop_saint_lucia = {
    2000: 156746, 2001: 158162, 2002: 159576, 2003: 161048,
    2004: 162560, 2005: 164103, 2006: 165659, 2007: 167235,
    2008: 168842, 2009: 170490, 2010: 172178, 2011: 173836,
    2012: 175321, 2013: 176552, 2014: 177539, 2015: 178312,
}

gdp_saint_lucia = {
    2000: 5123, 2001: 4853, 2002: 4985, 2003: 5283,
    2004: 5821, 2005: 6149, 2006: 6839, 2007: 7404,
    2008: 8007, 2009: 7532, 2010: 8141, 2011: 8527,
    2012: 8576, 2013: 8707, 2014: 9079, 2015: 9484,
}

mask = df["Country"] == "Saint Lucia"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_saint_lucia)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_saint_lucia)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


          Country  Year  population  gdp_per_capita
2160  Saint Lucia  2015    178312.0          9484.0
2161  Saint Lucia  2014    177539.0          9079.0
2162  Saint Lucia  2013    176552.0          8707.0
2163  Saint Lucia  2012    175321.0          8576.0
2164  Saint Lucia  2011    173836.0          8527.0
2165  Saint Lucia  2010    172178.0          8141.0
2166  Saint Lucia  2009    170490.0          7532.0
2167  Saint Lucia  2008    168842.0          8007.0
2168  Saint Lucia  2007    167235.0          7404.0
2169  Saint Lucia  2006    165659.0          6839.0
2170  Saint Lucia  2005    164103.0          6149.0
2171  Saint Lucia  2004    162560.0          5821.0
2172  Saint Lucia  2003    161048.0          5283.0
2173  Saint Lucia  2002    159576.0          4985.0
2174  Saint Lucia  2001    158162.0          4853.0
2175  Saint Lucia  2000    156746.0          5123.0


In [939]:
# Saint Vincent and the Grenadines
pop_svg = {
    2000: 113545, 2001: 113501, 2002: 113347, 2003: 113141,
    2004: 112835, 2005: 112450, 2006: 112014, 2007: 111531,
    2008: 111028, 2009: 110508, 2010: 109959, 2011: 109349,
    2012: 108736, 2013: 108135, 2014: 107535, 2015: 106960,
}

gdp_svg = {
    2000: 3769, 2001: 4071, 2002: 4303, 2003: 4500,
    2004: 4873, 2005: 5157, 2006: 5745, 2007: 6398,
    2008: 6599, 2009: 6464, 2010: 6552, 2011: 6528,
    2012: 6714, 2013: 7072, 2014: 7169, 2015: 7354,
}

mask = df["Country"] == "Saint Vincent and the Grenadines"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_svg)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_svg)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                               Country  Year  population  gdp_per_capita
2176  Saint Vincent and the Grenadines  2015    106960.0          7354.0
2177  Saint Vincent and the Grenadines  2014    107535.0          7169.0
2178  Saint Vincent and the Grenadines  2013    108135.0          7072.0
2179  Saint Vincent and the Grenadines  2012    108736.0          6714.0
2180  Saint Vincent and the Grenadines  2011    109349.0          6528.0
2181  Saint Vincent and the Grenadines  2010    109959.0          6552.0
2182  Saint Vincent and the Grenadines  2009    110508.0          6464.0
2183  Saint Vincent and the Grenadines  2008    111028.0          6599.0
2184  Saint Vincent and the Grenadines  2007    111531.0          6398.0
2185  Saint Vincent and the Grenadines  2006    112014.0          5745.0
2186  Saint Vincent and the Grenadines  2005    112450.0          5157.0
2187  Saint Vincent and the Grenadines  2004    112835.0          4873.0
2188  Saint Vincent and the Grenadines  2003    113

In [940]:
# Saudi Arabia
pop_saudi = {
    2000: 21547397, 2001: 22227315, 2002: 22904894, 2003: 23591659,
    2004: 24307673, 2005: 25069871, 2006: 25877544, 2007: 26711885,
    2008: 27517709, 2009: 28245671, 2010: 29411929, 2011: 30344057,
    2012: 31170051, 2013: 31940306, 2014: 32749848, 2015: 33695303,
}

mask = df["Country"] == "Saudi Arabia"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_saudi)

print(df.loc[mask, ["Country", "Year", "population"]])


           Country  Year  population
2224  Saudi Arabia  2015  33695303.0
2225  Saudi Arabia  2014  32749848.0
2226  Saudi Arabia  2013  31940306.0
2227  Saudi Arabia  2012  31170051.0
2228  Saudi Arabia  2011  30344057.0
2229  Saudi Arabia  2010  29411929.0
2230  Saudi Arabia  2009  28245671.0
2231  Saudi Arabia  2008  27517709.0
2232  Saudi Arabia  2007  26711885.0
2233  Saudi Arabia  2006  25877544.0
2234  Saudi Arabia  2005  25069871.0
2235  Saudi Arabia  2004  24307673.0
2236  Saudi Arabia  2003  23591659.0
2237  Saudi Arabia  2002  22904894.0
2238  Saudi Arabia  2001  22227315.0
2239  Saudi Arabia  2000  21547397.0


In [941]:
# Singapore
pop_singapore = {
    2000: 4027900, 2001: 4138000, 2002: 4176000, 2003: 4114800,
    2004: 4166700, 2005: 4265800, 2006: 4401400, 2007: 4588600,
    2008: 4839400, 2009: 4987600, 2010: 5076700, 2011: 5183700,
    2012: 5312400, 2013: 5399200, 2014: 5469700, 2015: 5535000,
}

mask = df["Country"] == "Singapore"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_singapore)

print(df.loc[mask, ["Country", "Year", "population"]])


        Country  Year  population
2304  Singapore  2015   5535000.0
2305  Singapore  2014   5469700.0
2306  Singapore  2013   5399200.0
2307  Singapore  2012   5312400.0
2308  Singapore  2011   5183700.0
2309  Singapore  2010   5076700.0
2310  Singapore  2009   4987600.0
2311  Singapore  2008   4839400.0
2312  Singapore  2007   4588600.0
2313  Singapore  2006   4401400.0
2314  Singapore  2005   4265800.0
2315  Singapore  2004   4166700.0
2316  Singapore  2003   4114800.0
2317  Singapore  2002   4176000.0
2318  Singapore  2001   4138000.0
2319  Singapore  2000   4027900.0


In [942]:
# Slovakia
pop_slovakia = {
    2000: 5388720, 2001: 5378867, 2002: 5376912, 2003: 5373374,
    2004: 5372280, 2005: 5372807, 2006: 5373054, 2007: 5374622,
    2008: 5379233, 2009: 5386406, 2010: 5391428, 2011: 5398384,
    2012: 5407579, 2013: 5413393, 2014: 5418649, 2015: 5423801,
}

gdp_slovakia = {
    2000: 3831,  2001: 3975,  2002: 4589,  2003: 6236,
    2004: 7908,  2005: 9045,  2006: 10471, 2007: 14249,
    2008: 17794, 2009: 16536, 2010: 16627, 2011: 18349,
    2012: 17392, 2013: 18131, 2014: 18634, 2015: 16339,
}

mask = df["Country"] == "Slovakia"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_slovakia)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_slovakia)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


       Country  Year  population  gdp_per_capita
2320  Slovakia  2015   5423801.0         16339.0
2321  Slovakia  2014   5418649.0         18634.0
2322  Slovakia  2013   5413393.0         18131.0
2323  Slovakia  2012   5407579.0         17392.0
2324  Slovakia  2011   5398384.0         18349.0
2325  Slovakia  2010   5391428.0         16627.0
2326  Slovakia  2009   5386406.0         16536.0
2327  Slovakia  2008   5379233.0         17794.0
2328  Slovakia  2007   5374622.0         14249.0
2329  Slovakia  2006   5373054.0         10471.0
2330  Slovakia  2005   5372807.0          9045.0
2331  Slovakia  2004   5372280.0          7908.0
2332  Slovakia  2003   5373374.0          6236.0
2333  Slovakia  2002   5376912.0          4589.0
2334  Slovakia  2001   5378867.0          3975.0
2335  Slovakia  2000   5388720.0          3831.0


In [943]:
# Somalia
pop_somalia = {
    2000: 8872254,  2001: 9150056,  2002: 9434606,  2003: 9726905,
    2004: 10028127, 2005: 10339261, 2006: 10660670, 2007: 10993605,
    2008: 11338321, 2009: 11694769, 2010: 12060277, 2011: 12430736,
    2012: 12810480, 2013: 13205953, 2014: 13616368, 2015: 14039814,
}

gdp_somalia = {
    2000: 386, 2001: 245, 2002: 230, 2003: 286,
    2004: 375, 2005: 440, 2006: 458, 2007: 480,
    2008: 507, 2009: 247, 2010: 219, 2011: 235,
    2012: 350, 2013: 393, 2014: 429, 2015: 446,
}

mask = df["Country"] == "Somalia"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_somalia)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_somalia)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


      Country  Year  population  gdp_per_capita
2368  Somalia  2015  14039814.0           446.0
2369  Somalia  2014  13616368.0           429.0
2370  Somalia  2013  13205953.0           393.0
2371  Somalia  2012  12810480.0           350.0
2372  Somalia  2011  12430736.0           235.0
2373  Somalia  2010  12060277.0           219.0
2374  Somalia  2009  11694769.0           247.0
2375  Somalia  2008  11338321.0           507.0
2376  Somalia  2007  10993605.0           480.0
2377  Somalia  2006  10660670.0           458.0
2378  Somalia  2005  10339261.0           440.0
2379  Somalia  2004  10028127.0           375.0
2380  Somalia  2003   9726905.0           286.0
2381  Somalia  2002   9434606.0           230.0
2382  Somalia  2001   9150056.0           245.0
2383  Somalia  2000   8872254.0           386.0


In [944]:
# The former Yugoslav republic of Macedonia
pop_macedonia = {
    2000: 2026350, 2001: 2034882, 2002: 2020157, 2003: 2022725,
    2004: 2016186, 2005: 2005330, 2006: 1994287, 2007: 1982933,
    2008: 1971493, 2009: 1958782, 2010: 1946298, 2011: 1937398,
    2012: 1929821, 2013: 1922716, 2014: 1917557, 2015: 1912430,
}

gdp_macedonia = {
    2000: 1862, 2001: 1823, 2002: 1989, 2003: 2445,
    2004: 2819, 2005: 3121, 2006: 3440, 2007: 4204,
    2008: 5026, 2009: 4800, 2010: 4833, 2011: 5417,
    2012: 5050, 2013: 5626, 2014: 5925, 2015: 5263,
}

mask = df["Country"] == "The former Yugoslav republic of Macedonia"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_macedonia)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_macedonia)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                                        Country  Year  population  \
2560  The former Yugoslav republic of Macedonia  2015   1912430.0   
2561  The former Yugoslav republic of Macedonia  2014   1917557.0   
2562  The former Yugoslav republic of Macedonia  2013   1922716.0   
2563  The former Yugoslav republic of Macedonia  2012   1929821.0   
2564  The former Yugoslav republic of Macedonia  2011   1937398.0   
2565  The former Yugoslav republic of Macedonia  2010   1946298.0   
2566  The former Yugoslav republic of Macedonia  2009   1958782.0   
2567  The former Yugoslav republic of Macedonia  2008   1971493.0   
2568  The former Yugoslav republic of Macedonia  2007   1982933.0   
2569  The former Yugoslav republic of Macedonia  2006   1994287.0   
2570  The former Yugoslav republic of Macedonia  2005   2005330.0   
2571  The former Yugoslav republic of Macedonia  2004   2016186.0   
2572  The former Yugoslav republic of Macedonia  2003   2022725.0   
2573  The former Yugoslav republic

In [945]:
# United Arab Emirates
pop_uae = {
    2000: 3275333, 2001: 3450137, 2002: 3651556, 2003: 3903346,
    2004: 4217605, 2005: 4588225, 2006: 5084013, 2007: 5835274,
    2008: 6788214, 2009: 7647794, 2010: 8270684, 2011: 8672475,
    2012: 8900453, 2013: 9006263, 2014: 9070867, 2015: 9154302,
}

mask = df["Country"] == "United Arab Emirates"
df.loc[mask, "population"] = df.loc[mask, "Year"].map(pop_uae)

print(df.loc[mask, ["Country", "Year", "population"]])


                   Country  Year  population
2720  United Arab Emirates  2015   9154302.0
2721  United Arab Emirates  2014   9070867.0
2722  United Arab Emirates  2013   9006263.0
2723  United Arab Emirates  2012   8900453.0
2724  United Arab Emirates  2011   8672475.0
2725  United Arab Emirates  2010   8270684.0
2726  United Arab Emirates  2009   7647794.0
2727  United Arab Emirates  2008   6788214.0
2728  United Arab Emirates  2007   5835274.0
2729  United Arab Emirates  2006   5084013.0
2730  United Arab Emirates  2005   4588225.0
2731  United Arab Emirates  2004   4217605.0
2732  United Arab Emirates  2003   3903346.0
2733  United Arab Emirates  2002   3651556.0
2734  United Arab Emirates  2001   3450137.0
2735  United Arab Emirates  2000   3275333.0


In [946]:
# United Kingdom of Great Britain and Northern Ireland
pop_uk = {
    2000: 58892514, 2001: 59119673, 2002: 59370479, 2003: 59647577,
    2004: 59987905, 2005: 60401257, 2006: 60846820, 2007: 61322463,
    2008: 61806995, 2009: 62276270, 2010: 62759456, 2011: 63258918,
    2012: 63700215, 2013: 64128273, 2014: 64602298, 2015: 65116219,
}

gdp_uk = {
    2000: 28291, 2001: 27515, 2002: 30111, 2003: 34324,
    2004: 40250, 2005: 42030, 2006: 44668, 2007: 50560,
    2008: 47562, 2009: 38710, 2010: 39663, 2011: 42121,
    2012: 42485, 2013: 43445, 2014: 47448, 2015: 45039,
}

mask = df["Country"] == "United Kingdom of Great Britain and Northern Ireland"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_uk)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_uk)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                                                Country  Year  population  \
2736  United Kingdom of Great Britain and Northern I...  2015  65116219.0   
2737  United Kingdom of Great Britain and Northern I...  2014  64602298.0   
2738  United Kingdom of Great Britain and Northern I...  2013  64128273.0   
2739  United Kingdom of Great Britain and Northern I...  2012  63700215.0   
2740  United Kingdom of Great Britain and Northern I...  2011  63258918.0   
2741  United Kingdom of Great Britain and Northern I...  2010  62759456.0   
2742  United Kingdom of Great Britain and Northern I...  2009  62276270.0   
2743  United Kingdom of Great Britain and Northern I...  2008  61806995.0   
2744  United Kingdom of Great Britain and Northern I...  2007  61322463.0   
2745  United Kingdom of Great Britain and Northern I...  2006  60846820.0   
2746  United Kingdom of Great Britain and Northern I...  2005  60401257.0   
2747  United Kingdom of Great Britain and Northern I...  2004  59987905.0   

In [947]:
# United Republic of Tanzania
pop_tanzania = {
    2000: 33499180, 2001: 34462754, 2002: 35463656, 2003: 36496164,
    2004: 37564285, 2005: 38674310, 2006: 39836664, 2007: 41059573,
    2008: 42353171, 2009: 43724113, 2010: 45178088, 2011: 46715625,
    2012: 48328634, 2013: 50006128, 2014: 51735745, 2015: 53501480,
}

gdp_tanzania = {
    2000: 402, 2001: 397, 2002: 403, 2003: 422,
    2004: 450, 2005: 483, 2006: 476, 2007: 543,
    2008: 676, 2009: 694, 2010: 737, 2011: 775,
    2012: 862, 2013: 963, 2014: 1023, 2015: 939,
}

mask = df["Country"] == "United Republic of Tanzania"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_tanzania)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_tanzania)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                          Country  Year  population  gdp_per_capita
2752  United Republic of Tanzania  2015  53501480.0           939.0
2753  United Republic of Tanzania  2014  51735745.0          1023.0
2754  United Republic of Tanzania  2013  50006128.0           963.0
2755  United Republic of Tanzania  2012  48328634.0           862.0
2756  United Republic of Tanzania  2011  46715625.0           775.0
2757  United Republic of Tanzania  2010  45178088.0           737.0
2758  United Republic of Tanzania  2009  43724113.0           694.0
2759  United Republic of Tanzania  2008  42353171.0           676.0
2760  United Republic of Tanzania  2007  41059573.0           543.0
2761  United Republic of Tanzania  2006  39836664.0           476.0
2762  United Republic of Tanzania  2005  38674310.0           483.0
2763  United Republic of Tanzania  2004  37564285.0           450.0
2764  United Republic of Tanzania  2003  36496164.0           422.0
2765  United Republic of Tanzania  2002  3546365

In [948]:
# United States of America
pop_usa = {
    2000: 282162411, 2001: 284968955, 2002: 287625193, 2003: 290107933,
    2004: 292805298, 2005: 295516599, 2006: 298379912, 2007: 301231207,
    2008: 304093966, 2009: 306771529, 2010: 309327143, 2011: 311583481,
    2012: 313877662, 2013: 316059947, 2014: 318386329, 2015: 320738994,
}

gdp_usa = {
    2000: 36330, 2001: 37134, 2002: 37998, 2003: 39490,
    2004: 41725, 2005: 44123, 2006: 46302, 2007: 48050,
    2008: 48570, 2009: 47195, 2010: 48651, 2011: 50066,
    2012: 51784, 2013: 53410, 2014: 55304, 2015: 57040,
}

mask = df["Country"] == "United States of America"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_usa)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_usa)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                       Country  Year   population  gdp_per_capita
2768  United States of America  2015  320738994.0         57040.0
2769  United States of America  2014  318386329.0         55304.0
2770  United States of America  2013  316059947.0         53410.0
2771  United States of America  2012  313877662.0         51784.0
2772  United States of America  2011  311583481.0         50066.0
2773  United States of America  2010  309327143.0         48651.0
2774  United States of America  2009  306771529.0         47195.0
2775  United States of America  2008  304093966.0         48570.0
2776  United States of America  2007  301231207.0         48050.0
2777  United States of America  2006  298379912.0         46302.0
2778  United States of America  2005  295516599.0         44123.0
2779  United States of America  2004  292805298.0         41725.0
2780  United States of America  2003  290107933.0         39490.0
2781  United States of America  2002  287625193.0         37998.0
2782  Unit

In [949]:
# Venezuela (Bolivarian Republic of)
pop_venezuela = {
    2000: 24427729, 2001: 24948476, 2002: 25445479, 2003: 25932761,
    2004: 26432447, 2005: 26941873, 2006: 27452365, 2007: 27947757,
    2008: 28420076, 2009: 28887707, 2010: 29362449, 2011: 29445419,
    2012: 29854238, 2013: 30276045, 2014: 30043908, 2015: 30563433,
}

gdp_venezuela = {
    2000: 4842,  2001: 5024,  2002: 3711,  2003: 3212,
    2004: 4249,  2005: 5404,  2006: 6722,  2007: 8337,
    2008: 11192, 2009: 11460, 2010: 13566, 2011: 10741,
    2012: 12755, 2013: 12290, 2014: 15662, 2015: 10568,
}

mask = df["Country"] == "Venezuela (Bolivarian Republic of)"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_venezuela)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_venezuela)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                                 Country  Year  population  gdp_per_capita
2832  Venezuela (Bolivarian Republic of)  2015  30563433.0         10568.0
2833  Venezuela (Bolivarian Republic of)  2014  30043908.0         15662.0
2834  Venezuela (Bolivarian Republic of)  2013  30276045.0         12290.0
2835  Venezuela (Bolivarian Republic of)  2012  29854238.0         12755.0
2836  Venezuela (Bolivarian Republic of)  2011  29445419.0         10741.0
2837  Venezuela (Bolivarian Republic of)  2010  29362449.0         13566.0
2838  Venezuela (Bolivarian Republic of)  2009  28887707.0         11460.0
2839  Venezuela (Bolivarian Republic of)  2008  28420076.0         11192.0
2840  Venezuela (Bolivarian Republic of)  2007  27947757.0          8337.0
2841  Venezuela (Bolivarian Republic of)  2006  27452365.0          6722.0
2842  Venezuela (Bolivarian Republic of)  2005  26941873.0          5404.0
2843  Venezuela (Bolivarian Republic of)  2004  26432447.0          4249.0
2844  Venezuela (Bolivari

In [950]:
# Viet Nam
pop_vietnam = {
    2000: 79001141, 2001: 80016193, 2002: 81020146, 2003: 82032301,
    2004: 83062820, 2005: 84108811, 2006: 85145285, 2007: 86117144,
    2008: 87030852, 2009: 87915016, 2010: 88771885, 2011: 89708866,
    2012: 90728949, 2013: 91777665, 2014: 92824375, 2015: 93870642,
}

gdp_vietnam = {
    2000: 390,  2001: 405,  2002: 440,  2003: 489,
    2004: 554,  2005: 699,  2006: 797,  2007: 919,
    2008: 1165, 2009: 1232, 2010: 1336, 2011: 1543,
    2012: 1755, 2013: 1908, 2014: 2052, 2015: 2103,
}

mask = df["Country"] == "Viet Nam"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_vietnam)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_vietnam)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


       Country  Year  population  gdp_per_capita
2848  Viet Nam  2015  93870642.0          2103.0
2849  Viet Nam  2014  92824375.0          2052.0
2850  Viet Nam  2013  91777665.0          1908.0
2851  Viet Nam  2012  90728949.0          1755.0
2852  Viet Nam  2011  89708866.0          1543.0
2853  Viet Nam  2010  88771885.0          1336.0
2854  Viet Nam  2009  87915016.0          1232.0
2855  Viet Nam  2008  87030852.0          1165.0
2856  Viet Nam  2007  86117144.0           919.0
2857  Viet Nam  2006  85145285.0           797.0
2858  Viet Nam  2005  84108811.0           699.0
2859  Viet Nam  2004  83062820.0           554.0
2860  Viet Nam  2003  82032301.0           489.0
2861  Viet Nam  2002  81020146.0           440.0
2862  Viet Nam  2001  80016193.0           405.0
2863  Viet Nam  2000  79001141.0           390.0


In [951]:
# Yemen
pop_yemen = {
    2000: 18628700, 2001: 19143433, 2002: 19660653, 2003: 20188712,
    2004: 20733406, 2005: 21320671, 2006: 21966298, 2007: 22641519,
    2008: 23329004, 2009: 24029589, 2010: 24743946, 2011: 25475610,
    2012: 26223391, 2013: 26984002, 2014: 27753304, 2015: 28516545,
}

gdp_yemen = {
    2000: 554,  2001: 550,  2002: 566,  2003: 590,
    2004: 688,  2005: 813,  2006: 889,  2007: 985,
    2008: 1171, 2009: 1059, 2010: 1257, 2011: 1313,
    2012: 1374, 2013: 1522, 2014: 1589, 2015: 1521,
}

mask = df["Country"] == "Yemen"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_yemen)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_yemen)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


     Country  Year  population  gdp_per_capita
2864   Yemen  2015  28516545.0          1521.0
2865   Yemen  2014  27753304.0          1589.0
2866   Yemen  2013  26984002.0          1522.0
2867   Yemen  2012  26223391.0          1374.0
2868   Yemen  2011  25475610.0          1313.0
2869   Yemen  2010  24743946.0          1257.0
2870   Yemen  2009  24029589.0          1059.0
2871   Yemen  2008  23329004.0          1171.0
2872   Yemen  2007  22641519.0           985.0
2873   Yemen  2006  21966298.0           889.0
2874   Yemen  2005  21320671.0           813.0
2875   Yemen  2004  20733406.0           688.0
2876   Yemen  2003  20188712.0           590.0
2877   Yemen  2002  19660653.0           566.0
2878   Yemen  2001  19143433.0           550.0
2879   Yemen  2000  18628700.0           554.0


In [952]:
# Syrian Arab Republic
gdp_syria = {
    2000: 1210, 2001: 1141, 2002: 1192, 2003: 1200,
    2004: 1343, 2005: 1501, 2006: 1712, 2007: 1974,
    2008: 2492, 2009: 2467, 2010: 2806, 2011: 2185,
    2012: 1850, 2013: 1500, 2014: 1200, 2015: 900,
}

mask = df["Country"] == "Syrian Arab Republic"
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_syria)

print(df.loc[mask, ["Country", "Year", "gdp_per_capita"]])


                   Country  Year  gdp_per_capita
2512  Syrian Arab Republic  2015           900.0
2513  Syrian Arab Republic  2014          1200.0
2514  Syrian Arab Republic  2013          1500.0
2515  Syrian Arab Republic  2012          1850.0
2516  Syrian Arab Republic  2011          2185.0
2517  Syrian Arab Republic  2010          2806.0
2518  Syrian Arab Republic  2009          2467.0
2519  Syrian Arab Republic  2008          2492.0
2520  Syrian Arab Republic  2007          1974.0
2521  Syrian Arab Republic  2006          1712.0
2522  Syrian Arab Republic  2005          1501.0
2523  Syrian Arab Republic  2004          1343.0
2524  Syrian Arab Republic  2003          1200.0
2525  Syrian Arab Republic  2002          1192.0
2526  Syrian Arab Republic  2001          1141.0
2527  Syrian Arab Republic  2000          1210.0


In [953]:
# Sao Tome and Principe
pop_stp = {
    2000: 144018, 2001: 146375, 2002: 149922, 2003: 153814,
    2004: 157705, 2005: 161639, 2006: 165638, 2007: 169700,
    2008: 173793, 2009: 177853, 2010: 181802, 2011: 185649,
    2012: 189439, 2013: 193148, 2014: 196767, 2015: 200291,
}

gdp_stp = {
    2000: 533,  2001: 489,  2002: 533,  2003: 621,
    2004: 663,  2005: 774,  2006: 805,  2007: 852,
    2008: 1082, 2009: 1128, 2010: 1049, 2011: 1229,
    2012: 1220, 2013: 1393, 2014: 1500, 2015: 1307,
}

mask = df["Country"] == "Sao Tome and Principe"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_stp)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_stp)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


                    Country  Year  population  gdp_per_capita
2208  Sao Tome and Principe  2015    200291.0          1307.0
2209  Sao Tome and Principe  2014    196767.0          1500.0
2210  Sao Tome and Principe  2013    193148.0          1393.0
2211  Sao Tome and Principe  2012    189439.0          1220.0
2212  Sao Tome and Principe  2011    185649.0          1229.0
2213  Sao Tome and Principe  2010    181802.0          1049.0
2214  Sao Tome and Principe  2009    177853.0          1128.0
2215  Sao Tome and Principe  2008    173793.0          1082.0
2216  Sao Tome and Principe  2007    169700.0           852.0
2217  Sao Tome and Principe  2006    165638.0           805.0
2218  Sao Tome and Principe  2005    161639.0           774.0
2219  Sao Tome and Principe  2004    157705.0           663.0
2220  Sao Tome and Principe  2003    153814.0           621.0
2221  Sao Tome and Principe  2002    149922.0           533.0
2222  Sao Tome and Principe  2001    146375.0           489.0
2223  Sa

In [954]:
# Papua New Guinea
pop_png = {
    2000: 5508297, 2001: 5683959, 2002: 5862316, 2003: 6045435,
    2004: 6233962, 2005: 6428293, 2006: 6628491, 2007: 6833124,
    2008: 7043029, 2009: 7259484, 2010: 7483184, 2011: 7713296,
    2012: 7949957, 2013: 8190560, 2014: 8437122, 2015: 8690854,
}

gdp_png = {
    2000: 636,  2001: 538,  2002: 506,  2003: 577,
    2004: 621,  2005: 744,  2006: 1238, 2007: 1371,
    2008: 1625, 2009: 1569, 2010: 1867, 2011: 2288,
    2012: 2635, 2013: 2561, 2014: 2723, 2015: 2485,
}

mask = df["Country"] == "Papua New Guinea"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_png)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_png)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


               Country  Year  population  gdp_per_capita
1968  Papua New Guinea  2015   8690854.0          2485.0
1969  Papua New Guinea  2014   8437122.0          2723.0
1970  Papua New Guinea  2013   8190560.0          2561.0
1971  Papua New Guinea  2012   7949957.0          2635.0
1972  Papua New Guinea  2011   7713296.0          2288.0
1973  Papua New Guinea  2010   7483184.0          1867.0
1974  Papua New Guinea  2009   7259484.0          1569.0
1975  Papua New Guinea  2008   7043029.0          1625.0
1976  Papua New Guinea  2007   6833124.0          1371.0
1977  Papua New Guinea  2006   6628491.0          1238.0
1978  Papua New Guinea  2005   6428293.0           744.0
1979  Papua New Guinea  2004   6233962.0           621.0
1980  Papua New Guinea  2003   6045435.0           577.0
1981  Papua New Guinea  2002   5862316.0           506.0
1982  Papua New Guinea  2001   5683959.0           538.0
1983  Papua New Guinea  2000   5508297.0           636.0


In [955]:
# Libya
pop_libya = {
    2000: 5355081, 2001: 5446145, 2002: 5537058, 2003: 5630132,
    2004: 5727614, 2005: 5830517, 2006: 5935165, 2007: 6036543,
    2008: 6132657, 2009: 6220133, 2010: 6291555, 2011: 6336634,
    2012: 6349949, 2013: 6321200, 2014: 6335014, 2015: 6418315,
}

gdp_libya = {
    2000: 7144,  2001: 6236,  2002: 3842,  2003: 4749,
    2004: 6106,  2005: 8502,  2006: 9800,  2007: 11947,
    2008: 14976, 2009: 10188, 2010: 12120, 2011: 5648,
    2012: 13639, 2013: 10480, 2014: 6478,  2015: 4166,
}

mask = df["Country"] == "Libya"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_libya)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_libya)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


     Country  Year  population  gdp_per_capita
1504   Libya  2015   6418315.0          4166.0
1505   Libya  2014   6335014.0          6478.0
1506   Libya  2013   6321200.0         10480.0
1507   Libya  2012   6349949.0         13639.0
1508   Libya  2011   6336634.0          5648.0
1509   Libya  2010   6291555.0         12120.0
1510   Libya  2009   6220133.0         10188.0
1511   Libya  2008   6132657.0         14976.0
1512   Libya  2007   6036543.0         11947.0
1513   Libya  2006   5935165.0          9800.0
1514   Libya  2005   5830517.0          8502.0
1515   Libya  2004   5727614.0          6106.0
1516   Libya  2003   5630132.0          4749.0
1517   Libya  2002   5537058.0          3842.0
1518   Libya  2001   5446145.0          6236.0
1519   Libya  2000   5355081.0          7144.0


In [956]:
# Iraq
pop_iraq = {
    2000: 24628858, 2001: 25274013, 2002: 25950593, 2003: 26673536,
    2004: 27412654, 2005: 28143152, 2006: 28833203, 2007: 29444792,
    2008: 30050490, 2009: 30774621, 2010: 31663465, 2011: 32657454,
    2012: 33696863, 2013: 34749431, 2014: 35785737, 2015: 36794449,
}

gdp_iraq = {
    2000: 880,  2001: 803,  2002: 770,  2003: 647,
    2004: 1391, 2005: 1833, 2006: 2382, 2007: 3044,
    2008: 4383, 2009: 3656, 2010: 4484, 2011: 5856,
    2012: 6601, 2013: 6903, 2014: 6743, 2015: 4884,
}

mask = df["Country"] == "Iraq"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_iraq)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_iraq)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


     Country  Year  population  gdp_per_capita
1232    Iraq  2015  36794449.0          4884.0
1233    Iraq  2014  35785737.0          6743.0
1234    Iraq  2013  34749431.0          6903.0
1235    Iraq  2012  33696863.0          6601.0
1236    Iraq  2011  32657454.0          5856.0
1237    Iraq  2010  31663465.0          4484.0
1238    Iraq  2009  30774621.0          3656.0
1239    Iraq  2008  30050490.0          4383.0
1240    Iraq  2007  29444792.0          3044.0
1241    Iraq  2006  28833203.0          2382.0
1242    Iraq  2005  28143152.0          1833.0
1243    Iraq  2004  27412654.0          1391.0
1244    Iraq  2003  26673536.0           647.0
1245    Iraq  2002  25950593.0           770.0
1246    Iraq  2001  25274013.0           803.0
1247    Iraq  2000  24628858.0           880.0


In [957]:
# Eritrea
pop_eritrea = {
    2000: 2247031, 2001: 2312079, 2002: 2392978, 2003: 2493903,
    2004: 2598284, 2005: 2661215, 2006: 2703502, 2007: 2743222,
    2008: 2816327, 2009: 2887885, 2010: 2945186, 2011: 2998485,
    2012: 3036988, 2013: 3074360, 2014: 3095173, 2015: 3105546,
}

gdp_eritrea = {
    2000: 244, 2001: 193, 2002: 176, 2003: 230,
    2004: 298, 2005: 320, 2006: 378, 2007: 440,
    2008: 420, 2009: 450, 2010: 510, 2011: 550,
    2012: 600, 2013: 650, 2014: 680, 2015: 700,
}

mask = df["Country"] == "Eritrea"
df.loc[mask, "population"]     = df.loc[mask, "Year"].map(pop_eritrea)
df.loc[mask, "gdp_per_capita"] = df.loc[mask, "Year"].map(gdp_eritrea)

print(df.loc[mask, ["Country", "Year", "population", "gdp_per_capita"]])


     Country  Year  population  gdp_per_capita
848  Eritrea  2015   3105546.0           700.0
849  Eritrea  2014   3095173.0           680.0
850  Eritrea  2013   3074360.0           650.0
851  Eritrea  2012   3036988.0           600.0
852  Eritrea  2011   2998485.0           550.0
853  Eritrea  2010   2945186.0           510.0
854  Eritrea  2009   2887885.0           450.0
855  Eritrea  2008   2816327.0           420.0
856  Eritrea  2007   2743222.0           440.0
857  Eritrea  2006   2703502.0           378.0
858  Eritrea  2005   2661215.0           320.0
859  Eritrea  2004   2598284.0           298.0
860  Eritrea  2003   2493903.0           230.0
861  Eritrea  2002   2392978.0           176.0
862  Eritrea  2001   2312079.0           193.0
863  Eritrea  2000   2247031.0           244.0


## 2.5 Función para missings

In [958]:
# ── Función Maestra para imputar Missings ──────────────────────────────────
def impute_missing(df, col):
    def _impute(group):
        group = group.sort_values("Year").copy()
        for idx in group[group[col].isna()].index:
            year = group.loc[idx, "Year"]
            if year == 2015:
                ref = group.loc[group["Year"] == 2014, col].values
                if len(ref) and not np.isnan(ref[0]):
                    group.loc[idx, col] = ref[0]
            elif year == 2000:
                ref = group.loc[group["Year"] == 2001, col].values
                if len(ref) and not np.isnan(ref[0]):
                    group.loc[idx, col] = ref[0]
            else:
                prev = group.loc[group["Year"] == year - 1, col].values
                nxt  = group.loc[group["Year"] == year + 1, col].values
                vals = [v for v in [prev[0] if len(prev) else np.nan,
                                     nxt[0]  if len(nxt)  else np.nan]
                        if not np.isnan(v)]
                if vals:
                    group.loc[idx, col] = np.mean(vals)
        return group
    return df.groupby("Country", group_keys=False).apply(_impute)

In [959]:
# Imputamos como nulos los ceros de la columna health_exp_pct_gdp 
df["health_exp_pct_gdp"] = df["health_exp_pct_gdp"].replace(0, np.nan)
print(f"Zeros reemplazados. Missings en health_exp_pct_gdp: {df['health_exp_pct_gdp'].isna().sum()}")

df = impute_missing(df, "health_exp_pct_gdp")
print(f"Missings restantes en health_exp_pct_gdp: {df['health_exp_pct_gdp'].isna().sum()}")
df

Zeros reemplazados. Missings en health_exp_pct_gdp: 593
Missings restantes en health_exp_pct_gdp: 423


,Year,life_expectancy,adult_mortality,infant_deaths,alcohol,health_exp_pct_gdp,hepatitis_b,measles,bmi,under5_deaths,polio,gov_health_exp_pct,diphtheria,hiv_aids,gdp_per_capita,population,thinness_10_19,thinness_5_9,hdi_income,schooling,develop_status
15,2000,54.8,321.0,88,0.01,10.424960,62.0,6532,12.2,122,24.0,8.20,24.0,0.1,114.560000,293756.0,2.3,2.5,0.338,5.5,0
14,2001,55.3,316.0,88,0.01,10.574728,63.0,8762,12.6,122,35.0,7.80,33.0,0.1,117.496980,2966463.0,2.1,2.4,0.340,5.9,0
13,2002,56.2,3.0,88,0.01,16.887351,64.0,2486,13.0,122,36.0,7.76,36.0,0.1,187.845950,21979923.0,19.9,2.2,0.341,6.2,0
12,2003,56.7,295.0,87,0.01,11.089053,65.0,798,13.4,122,41.0,8.82,41.0,0.1,198.728544,2364851.0,19.7,19.9,0.373,6.5,0
11,2004,57.0,293.0,87,0.02,15.296066,67.0,466,13.8,120,5.0,8.79,5.0,0.1,219.141353,24118979.0,19.5,19.7,0.381,6.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2900,2011,54.9,464.0,28,6.00,63.750530,94.0,0,29.9,42,93.0,6.31,93.0,13.3,839.927936,14386649.0,6.8,6.7,0.452,10.1,0
2899,2012,56.6,429.0,26,6.09,92.602336,97.0,0,3.3,39,95.0,6.69,95.0,8.8,955.648466,1471826.0,6.5,6.4,0.464,9.8,0
2898,2013,58.0,399.0,25,6.39,10.666707,95.0,0,3.8,36,95.0,6.88,95.0,6.8,111.227396,155456.0,6.2,6.0,0.488,10.4,0
2897,2014,59.2,371.0,23,6.50,10.822595,91.0,0,31.3,34,92.0,6.44,91.0,6.3,127.474620,15411675.0,5.9,5.7,0.498,10.3,0
